<a href="https://colab.research.google.com/github/usama488/bioinformatics-analysis/blob/main/Drug_Discovery_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Drug Discovery using AI

# Project 9 Bioinformatics

Is notebook mein hum **cheminformatics + machine learning** use kar ke molecules ki **drug-likeness aur predicted bioactivity** assess karain gay — molecular descriptors (RDKit) se features nikal kar ek AI model banayein gay jo naye molecules ko **"Promising Candidate"** ya **"Unlikely Candidate"** classify kare.

# **Pipeline:**
# 1. RDKit setup — molecular descriptor calculation
# 2. Real drug molecules ka dataset (SMILES) banana
# 3. Descriptor extraction (Molecular Weight, LogP, TPSA, H-bond donors/acceptors, etc.)
# 4. Lipinski's Rule of Five compliance check
# 5. Interactive Plotly visualizations
# 6. ML model training — bioactivity/candidate prediction
# 7. **Runtime cell** — apna khud ka SMILES daal kar direct predict karein + molecule structure dekhein


## 1. Setup & Imports

In [1]:
# RDKit install (Colab mein zaroori hai — pehli dafa run karne par kernel restart maang sakta hai)
!pip install -q rdkit

import numpy as np
import pandas as pd
import base64
from io import BytesIO

from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, Lipinski, Crippen

import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix, roc_curve

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

np.random.seed(42)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 34.4 MB/s eta 0:00:00


## 2. Molecule Dataset (Real Drug SMILES)

# Hum kuch **real approved drugs** ke SMILES strings use kar rahe hain — chhotay "drug-like" molecules aur bariay/complex molecules dono shamil hain, taake diverse chemical space cover ho.


In [2]:
molecules = {
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "Ibuprofen": "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O",
    "Paracetamol": "CC(=O)NC1=CC=C(C=C1)O",
    "Caffeine": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "Metformin": "CN(C)C(=N)NC(=N)N",
    "Warfarin": "CC(=O)CC(c1ccccc1)c1c(O)c2ccccc2oc1=O",
    "Naproxen": "COc1ccc2cc(ccc2c1)C(C)C(=O)O",
    "Diphenhydramine": "CN(C)CCOC(c1ccccc1)c1ccccc1",
    "Loratadine": "CCOC(=O)N1CCC(=C2c3ccc(Cl)cc3CCc3cccnc32)CC1",
    "Sildenafil": "CCCc1nn(C)c2c1nc(nc2=O)c1cc(ccc1OCC)S(=O)(=O)N1CCN(C)CC1",
    "Losartan": "CCCCc1nc(Cl)c(CO)n1Cc1ccc(cc1)-c1ccccc1-c1nnn[nH]1",
    "Metoprolol": "COCCc1ccc(OCC(O)CNC(C)C)cc1",
    "Ranitidine": "CNC(=C[N+](=O)[O-])NCCSCc1ccc(o1)CN(C)C",
    "Diazepam": "CN1c2ccc(Cl)cc2C(=NCC1=O)c1ccccc1",
    "Omeprazole": "CC1=CN=C(C(=C1OC)C)CS(=O)c1nc2ccc(OC)cc2[nH]1",
    "Simvastatin": "CCC(C)(C)C(=O)OC1CC(C)C=C2C=CC(C)C(CCC3CC(O)CC(=O)O3)C12",
    "Atorvastatin": "CC(C)c1c(C(=O)Nc2ccccc2)c(-c2ccccc2)c(-c2ccc(F)cc2)n1CCC(O)CC(O)CC(=O)O",
    "Amoxicillin": "CC1(C)S[C@@H]2[C@H](NC(=O)[C@H](N)c3ccc(O)cc3)C(=O)N2[C@H]1C(=O)O",
    "Penicillin G": "CC1(C)S[C@@H]2[C@H](NC(=O)Cc3ccccc3)C(=O)N2[C@H]1C(=O)O",
    "Ciprofloxacin": "OC(=O)c1cn(C2CC2)c2cc(N3CCNCC3)c(F)cc2c1=O",
    "Cyclosporine A": "CC[C@H]1NC(=O)[C@H]([C@H](O)[C@H](C)C\\C=C\\C)N(C)C(=O)[C@H](C(C)C)N(C)C(=O)[C@H](CC(C)C)N(C)C(=O)[C@H](CC(C)C)N(C)C(=O)[C@H](C)NC(=O)[C@H](C)NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@H](C(C)C)NC(=O)[C@@H]1CC(C)C",
    "Vancomycin": "CN[C@H]1[C@@H](O)[C@@H](O)[C@H](C)O[C@H]1O[C@@H]1[C@H](OC2=CC3=CC(NC(=O)[C@@H]4C(=O)N[C@@H](C(=O)N[C@H]5C(=O)N[C@H](C(=O)N[C@@H](CC(N)=O)C(=O)N[C@H](C(O)=O)c6cc(O)cc(O)c6-c6cc4ccc6O)[C@H](O)c4ccc(O)c(c4)Oc4cc3cc(Cl)c4O)c3ccc(O)c(Cl)c3)c(=O)cc2O)O[C@H](CO)[C@@H](O)[C@@H]1O",
    "Paclitaxel": "CC1=C2C(C(=O)C3(C(CC4C(C3C(C(C2(C)C)(CC1OC(=O)C(C(C5=CC=CC=C5)NC(=O)C6=CC=CC=C6)O)O)OC(=O)C7=CC=CC=C7)(CO4)OC(=O)C)O)C)OC(=O)C",
    "Rifampin": "CC1C=CC=C(C)C(=O)NC2=C(C(=O)C3=C(O)C4=C(C(=C3C2=O)O)C2(C)Oc3c(C)c(O)c(NC=NC4=O)c(O)c3C(C=C(C)C1O)N2)C",
    "Erythromycin": "CCC1OC(=O)C(C)C(OC2CC(C)(OC)C(O)C(C)O2)C(C)C(OC3OC(C)CC(N(C)C)C3O)C(C)(O)CC(C)C(=O)C(C)C(O)C1(C)O",
    "Insulin-like Peptide": "NCC(C(=O)NC(CC(=O)N)C(=O)NC(CC1=CC=CC=C1)C(=O)NC(CO)C(=O)NC(CC(=O)O)C(=O)NC(C)C(=O)NC(CCCCN)C(=O)O)N",
    "Tetracycline": "CC1(C2CC3C(C(=O)C(=C(C3(C(=O)C2=C(C4=C1C=CC=C4O)O)O)O)C(=O)N)N(C)C)O",
    "Digoxin": "CC1OC(OC2C(O)CC(OC3CC(O)C(OC4CCC5(C)C(CCC6C5CCC5(C)C6CCC6(C)OC(=O)C=C56)C4)OC3C)OC2C)CC(O)C1O",
}

print(f"Total reference molecules: {len(molecules)}")


Total reference molecules: 28


## 3. Extract Molecular Descriptors

In [3]:
def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return {
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "TPSA": Descriptors.TPSA(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "RotatableBonds": Descriptors.NumRotatableBonds(mol),
        "AromaticRings": Descriptors.NumAromaticRings(mol),
        "NumRings": Descriptors.RingCount(mol),
        "HeavyAtoms": Descriptors.HeavyAtomCount(mol),
        "MolarRefractivity": Crippen.MolMR(mol),
    }

rows = []
for name, smi in molecules.items():
    desc = compute_descriptors(smi)
    if desc:
        desc["name"] = name
        desc["smiles"] = smi
        rows.append(desc)

mol_df = pd.DataFrame(rows)
print(f"Descriptors computed for {len(mol_df)} molecules")
mol_df.head()


Descriptors computed for 26 molecules


[15:19:49] Can't kekulize mol.  Unkekulized atoms: 3 4 7 8 9 10 11
[15:19:49] Can't kekulize mol.  Unkekulized atoms: 3 4 7 8 9 10 11
[15:19:49] SMILES Parse Error: unclosed ring for input: 'CN[C@H]1[C@@H](O)[C@@H](O)[C@H](C)O[C@H]1O[C@@H]1[C@H](OC2=CC3=CC(NC(=O)[C@@H]4C(=O)N[C@@H](C(=O)N[C@H]5C(=O)N[C@H](C(=O)N[C@@H](CC(N)=O)C(=O)N[C@H](C(O)=O)c6cc(O)cc(O)c6-c6cc4ccc6O)[C@H](O)c4ccc(O)c(c4)Oc4cc3cc(Cl)c4O)c3ccc(O)c(Cl)c3)c(=O)cc2O)O[C@H](CO)[C@@H](O)[C@@H]1O'


,MolWt,LogP,TPSA,HBD,HBA,RotatableBonds,AromaticRings,NumRings,HeavyAtoms,MolarRefractivity,name,smiles
0,180.159,1.31010,63.60,1,3,2,1,1,13,44.7103,Aspirin,CC(=O)OC1=CC=CC=C1C(=O)O
1,206.285,3.07320,37.30,1,1,4,1,1,15,61.0348,Ibuprofen,CC(C)CC1=CC=C(C=C1)C(C)C(=O)O
2,151.165,1.35060,49.33,2,2,1,1,1,11,42.4105,Paracetamol,CC(=O)NC1=CC=C(C=C1)O
3,194.194,-1.02930,61.82,0,3,0,2,2,14,51.1960,Caffeine,CN1C=NC2=C1C(=O)N(C(=O)N2C)C
4,129.167,-1.03416,88.99,4,2,0,0,0,9,36.4635,Metformin,CN(C)C(=N)NC(=N)N


## 4. Lipinski's Rule of Five (Drug-likeness Check)

In [4]:
def lipinski_pass(row):
    violations = 0
    if row["MolWt"] > 500: violations += 1
    if row["LogP"] > 5: violations += 1
    if row["HBD"] > 5: violations += 1
    if row["HBA"] > 10: violations += 1
    return violations

mol_df["Ro5_violations"] = mol_df.apply(lipinski_pass, axis=1)
mol_df["Ro5_compliant"] = mol_df["Ro5_violations"] <= 1  # standard tolerance: max 1 violation allowed

print(f"Ro5-compliant molecules: {mol_df['Ro5_compliant'].sum()} / {len(mol_df)}")
mol_df[["name", "MolWt", "LogP", "HBD", "HBA", "Ro5_violations", "Ro5_compliant"]].sort_values("Ro5_violations")


Ro5-compliant molecules: 20 / 26


,name,MolWt,LogP,HBD,HBA,Ro5_violations,Ro5_compliant
0,Aspirin,180.159,1.31010,1,3,0,True
1,Ibuprofen,206.285,3.07320,1,1,0,True
2,Paracetamol,151.165,1.35060,2,2,0,True
3,Caffeine,194.194,-1.02930,0,3,0,True
4,Metformin,129.167,-1.03416,4,2,0,True
5,Warfarin,308.333,3.60960,1,4,0,True
6,Naproxen,230.263,3.03650,1,2,0,True
7,Diphenhydramine,255.361,3.35420,0,2,0,True
8,Loratadine,382.891,4.88780,0,3,0,True
9,Losartan,422.920,4.26680,2,5,0,True


## 5. Simulate Bioactivity Labels



**Real project ke liye:** ChEMBL/PubChem se real assay data (IC50, Ki, activity labels) download kar ke `mol_df["active"]` column replace karein.


In [5]:
rng = np.random.default_rng(42)

optimal_logp_score = 1 - np.abs(mol_df["LogP"] - 2.5) / 5
optimal_tpsa_score = 1 - np.abs(mol_df["TPSA"] - 70) / 150
ro5_score = mol_df["Ro5_compliant"].astype(float)

bioactivity_prob = (
    0.35 * ro5_score.clip(0, 1) +
    0.30 * optimal_logp_score.clip(0, 1) +
    0.20 * optimal_tpsa_score.clip(0, 1) +
    0.15 * rng.random(len(mol_df))
)
mol_df["active"] = (bioactivity_prob > np.median(bioactivity_prob)).astype(int)
mol_df["activity_label"] = mol_df["active"].map({1: "Promising Candidate", 0: "Unlikely Candidate"})

print(mol_df["activity_label"].value_counts())


activity_label
Promising Candidate    13
Unlikely Candidate     13
Name: count, dtype: int64


## 6. Interactive Visualizations

In [6]:
fig = px.scatter(
    mol_df, x="MolWt", y="LogP", color="activity_label", size="TPSA",
    hover_name="name", hover_data=["HBD", "HBA", "Ro5_violations"],
    title="Chemical Space: Molecular Weight vs LogP (bubble size = TPSA)",
    template="plotly_white",
    color_discrete_map={"Promising Candidate": "#43AA8B", "Unlikely Candidate": "#E63946"}
)
fig.add_vline(x=500, line_dash="dash", line_color="gray", annotation_text="MW=500 (Ro5 limit)")
fig.add_hline(y=5, line_dash="dash", line_color="gray", annotation_text="LogP=5 (Ro5 limit)")
fig.update_layout(height=550)
fig.show()


In [7]:
fig = px.pie(mol_df, names="activity_label", title="Predicted Candidate Distribution", hole=0.45,
             color="activity_label", color_discrete_map={"Promising Candidate": "#43AA8B", "Unlikely Candidate": "#E63946"})
fig.update_layout(height=400)
fig.show()

fig2 = px.parallel_coordinates(
    mol_df, dimensions=["MolWt", "LogP", "TPSA", "HBD", "HBA", "RotatableBonds"],
    color="active", color_continuous_scale=["#E63946", "#43AA8B"],
    title="Molecular Descriptor Profile — Parallel Coordinates"
)
fig2.update_layout(height=500)
fig2.show()


## 7. Train Bioactivity Prediction Model

In [8]:
feature_cols = ["MolWt", "LogP", "TPSA", "HBD", "HBA", "RotatableBonds", "AromaticRings", "NumRings", "HeavyAtoms", "MolarRefractivity"]
X = mol_df[feature_cols]
y = mol_df["active"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

drug_scaler = StandardScaler()
X_train_scaled = drug_scaler.fit_transform(X_train)
X_test_scaled = drug_scaler.transform(X_test)

drug_clf = RandomForestClassifier(n_estimators=300, random_state=42)
drug_clf.fit(X_train_scaled, y_train)

preds = drug_clf.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, preds):.3f}")
print(classification_report(y_test, preds, target_names=["Unlikely Candidate", "Promising Candidate"], zero_division=0))


Accuracy: 1.000
                     precision    recall  f1-score   support

 Unlikely Candidate       1.00      1.00      1.00         4
Promising Candidate       1.00      1.00      1.00         3

           accuracy                           1.00         7
          macro avg       1.00      1.00      1.00         7
       weighted avg       1.00      1.00      1.00         7



In [9]:
importances = pd.Series(drug_clf.feature_importances_, index=feature_cols).sort_values(ascending=False)
fig = px.bar(importances, orientation='h', title="Feature Importance — Bioactivity Prediction",
             labels={"value": "Importance", "index": "Descriptor"}, template="plotly_white",
             color=importances.values, color_continuous_scale="Viridis")
fig.update_layout(height=450, showlegend=False, yaxis={'categoryorder': 'total ascending'})
fig.show()


## 8.  Runtime Prediction — Apna Molecule (SMILES) Direct Input Karein




In [11]:
smiles_box = widgets.Text(
    value="CC(=O)OC1=CC=CC=C1C(=O)O",
    description="SMILES:",
    placeholder="e.g. CC(=O)OC1=CC=CC=C1C(=O)O (Aspirin)",
    style={'description_width': '80px'},
    layout=widgets.Layout(width='550px')
)
predict_btn = widgets.Button(description=" Predict Karein", button_style='success',
                              layout=widgets.Layout(width='220px', height='38px'))
out = widgets.Output()

def mol_to_base64_img(mol, size=(280, 220)):
    img = Draw.MolToImage(mol, size=size)
    buf = BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")

def render_drug_result(label, proba, desc, ro5_violations, img_b64):
    color = "#43AA8B" if label == "Promising Candidate" else "#E63946"
    emoji = "" if label == "Promising Candidate" else ""
    conf = proba[1]*100 if label == "Promising Candidate" else proba[0]*100
    ro5_status = " Passes Ro5" if ro5_violations <= 1 else f" {ro5_violations} Ro5 violations"

    html = f"""
    <div style="display:flex; gap:20px; border:2px solid {color}; border-radius:12px; padding:18px; margin-top:12px; font-family:sans-serif; background:#fafafa;">
        <img src="data:image/png;base64,{img_b64}" style="border-radius:8px; border:1px solid #ddd;"/>
        <div style="flex:1;">
            <div style="font-size:21px; font-weight:700; color:{color};">{emoji} {label}</div>
            <div style="font-size:14px; margin-top:6px;">Confidence: <b>{conf:.2f}%</b></div>
            <div style="margin-top:8px; height:12px; width:100%; background:#e0e0e0; border-radius:6px; overflow:hidden;">
                <div style="height:100%; width:{proba[1]*100:.1f}%; background:linear-gradient(90deg,#E63946,#43AA8B);"></div>
            </div>
            <div style="font-size:12px; color:#555; margin-top:10px;">
                MW: {desc['MolWt']:.1f} | LogP: {desc['LogP']:.2f} | TPSA: {desc['TPSA']:.1f}<br>
                HBD: {desc['HBD']} | HBA: {desc['HBA']} | Rotatable Bonds: {desc['RotatableBonds']}
            </div>
            <div style="font-size:13px; margin-top:8px; font-weight:600;">{ro5_status}</div>
        </div>
    </div>
    """
    display(HTML(html))

def on_predict(b):
    with out:
        clear_output()
        smi = smiles_box.value.strip()
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            print(" Invalid SMILES string — dobara check karein.")
            return

        desc = compute_descriptors(smi)
        row_df = pd.DataFrame([desc])[feature_cols]
        row_scaled = drug_scaler.transform(row_df)
        pred = drug_clf.predict(row_scaled)[0]
        proba = drug_clf.predict_proba(row_scaled)[0]
        label = "Promising Candidate" if pred == 1 else "Unlikely Candidate"

        violations = sum([desc["MolWt"] > 500, desc["LogP"] > 5, desc["HBD"] > 5, desc["HBA"] > 10])
        img_b64 = mol_to_base64_img(mol)

        render_drug_result(label, proba, desc, violations, img_b64)

predict_btn.on_click(on_predict)

display(smiles_box)
display(predict_btn)
display(out)


Text(value='CC(=O)OC1=CC=CC=C1C(=O)O', description='SMILES:', layout=Layout(width='550px'), placeholder='e.g. …

Button(button_style='success', description=' Predict Karein', layout=Layout(height='38px', width='220px'), sty…

Output()